In [4]:
import pandas as pd
import numpy as np

# --- Lire le fichier Excel (.xls) ---

df = pd.read_csv("/Users/serdarvarol/Desktop/PVL_L3-25_26/S2/sciDonne_L3/projet_vitamin_IA/data_csv_equilibree/vitamin_balanced_50.csv")

# --- Paramètre de la moyenne glissante ---
window = 10

# --- Colonnes numériques uniquement (on calcule mean/median/mode dessus) ---
num_cols = df.select_dtypes(include="number").columns.tolist()

# --- Fonctions de lissage ---
for col in num_cols:
    # Moyenne glissante
    df[f"{col}_ma{window}"] = df[col].rolling(window=window, min_periods=1).mean()
    # Médiane glissante
    df[f"{col}_median{window}"] = df[col].rolling(window=window, min_periods=1).median()
    # Mode glissant (valeur la plus fréquente sur la fenêtre)
    df[f"{col}_mode{window}"] = df[col].rolling(window=window, min_periods=1).apply(
        lambda x: x.mode().iloc[0] if len(x.mode()) else np.nan,
        raw=False
    )

from IPython.display import display
display(df)

# --- Sauvegarder le fichier final ---
output_path = "data2_with_ma_medvZZZz.csv"
df.to_csv(output_path, index=False)

output_path


,age,gender,bmi,smoking_status,alcohol_consumption,exercise_level,diet_type,sun_exposure,vitamin_a_percent_rda,vitamin_c_percent_rda,...,has_numbness_tingling_mode10,has_memory_problems_ma10,has_memory_problems_median10,has_memory_problems_mode10,has_pale_skin_ma10,has_pale_skin_median10,has_pale_skin_mode10,has_multiple_deficiencies_ma10,has_multiple_deficiencies_median10,has_multiple_deficiencies_mode10
0,20,Male,24.3,Former,Moderate,Sedentary,Vegan,Moderate,44.1,37.1,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
1,41,Female,22.2,Never,Moderate,Sedentary,Vegetarian,Low,45.6,81.9,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
2,40,Female,22.8,Current,Heavy,Light,Vegan,Low,32.3,101.4,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
3,25,Female,16.1,Current,Moderate,Moderate,Vegetarian,Low,23.0,71.7,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
4,56,Male,31.2,Current,Moderate,Light,Vegan,Moderate,48.7,49.2,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
245,28,Male,24.9,Current,Heavy,Active,Omnivore,High,26.6,66.4,...,0.0,0.4,0.0,0.0,0.2,0.0,0.0,0.9,1.0,1.0
246,56,Male,22.5,Never,Heavy,Moderate,Omnivore,Low,30.6,72.2,...,0.0,0.3,0.0,0.0,0.2,0.0,0.0,0.8,1.0,1.0
247,84,Male,25.2,Former,Heavy,Sedentary,Vegetarian,High,23.7,49.4,...,0.0,0.3,0.0,0.0,0.2,0.0,0.0,0.8,1.0,1.0
248,72,Male,27.3,Former,Heavy,Light,Vegetarian,Moderate,28.8,52.4,...,0.0,0.3,0.0,0.0,0.3,0.0,0.0,0.8,1.0,1.0


'data2_with_ma_medvZZZz.csv'

In [6]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1) Lire ton fichier
file_path = "data2_with_ma_medvZZZz.csv"  # mets le bon chemin si besoin
df = pd.read_csv(file_path)

# 2) Features (comme sur ta capture)
features = [
    "vitamin_a_percent_rda",
    "hemoglobin_g_dl",
    "vitamin_c_percent_rda",
    "vitamin_e_percent_rda",
    "vitamin_b12_percent_rda",
    "vitamin_d_percent_rda",
    "folate_percent_rda",
    "calcium_percent_rda",
    "iron_percent_rda",
    "serum_vitamin_d_ng_ml",
    "serum_vitamin_b12_pg_ml",
    "serum_folate_ng_ml",
    "has_muscle_weakness",
    "has_night_blindness",
    "has_bleeding_gums",
    "has_fatigue",
    "has_bone_pain",
    "has_numbness_tingling",
    "has_memory_problems",
    "has_pale_skin",
    "diet_type"
]

target = "disease_diagnosis"

# 3) Vérifier que les colonnes existent
missing = [c for c in features + [target] if c not in df.columns]
if missing:
    raise ValueError(f"Colonnes manquantes dans ton CSV : {missing}")

# 4) X et y
X = df[features]
y = df[target]

# 5) Colonnes numériques / catégorielles
num_cols = X.select_dtypes(include="number").columns.tolist()
cat_cols = X.select_dtypes(exclude="number").columns.tolist()

# 6) Pré-traitement (simple)
# - Numériques : remplacer NaN par médiane
# - Catégorielles : remplacer NaN par valeur la plus fréquente + OneHot
preprocess = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), num_cols),
        ("cat", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_cols),
    ]
)

# 7) Modèle Random Forest (simple)
model = RandomForestClassifier(n_estimators=200, random_state=42)

# 8) Pipeline complet
clf = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", model)
])

# 9) Train / Test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.05, random_state=42, stratify=y
)

# 10) Entraîner + prédire
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

# 11) Résultats
print("Accuracy =", accuracy_score(y_test, y_pred))
print("\nRapport détaillé :\n", classification_report(y_test, y_pred))
print("\nMatrice de confusion :\n", confusion_matrix(y_test, y_pred))


Accuracy = 1.0

Rapport détaillé :
                       precision    recall  f1-score   support

              Anemia       1.00      1.00      1.00         3
             Healthy       1.00      1.00      1.00         2
     Night_Blindness       1.00      1.00      1.00         3
Rickets_Osteomalacia       1.00      1.00      1.00         3
              Scurvy       1.00      1.00      1.00         2

            accuracy                           1.00        13
           macro avg       1.00      1.00      1.00        13
        weighted avg       1.00      1.00      1.00        13


Matrice de confusion :
 [[3 0 0 0 0]
 [0 2 0 0 0]
 [0 0 3 0 0]
 [0 0 0 3 0]
 [0 0 0 0 2]]
